In [ ]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie_short.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

In [ ]:
data_gdynia_long_categories

1.

In [ ]:
dtab1 = (
    data_gdynia_long_categories.groupby(["addr_street", "category"])
    .size()
    .unstack(fill_value=0)
)
dtab2 = pd.pivot_table(
    data_gdynia_long_categories,
    index="addr_street",
    columns="category",
    values="function_clean",
    aggfunc="size",
    fill_value=0,
)
dtab3 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
)

In [ ]:
pd.testing.assert_frame_equal(dtab1, dtab2)
pd.testing.assert_frame_equal(dtab2, dtab3)

assert (dtab1 == 0).sum(axis=None) == 443
assert dtab1.sum(axis=None) == 1124
assert dtab1.sum(axis=1).loc["Świętojańska"] == 229

2.

In [ ]:
dtab21 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
    normalize="index",
    margins=True,
)
dtab22 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
    normalize="columns",
)
dtab23 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
    normalize="all",
)

In [ ]:
assert np.isclose(dtab21.sum(axis=1), 1).all()
assert np.isclose(dtab22.sum(axis=0), 1).all()
assert np.isclose(dtab23.sum().sum(), 1)

In [ ]:
dtab22.style.background_gradient(cmap="Blues", axis="index").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

3.

In [ ]:
# def add_building_features(df_short,df_long):

# merge = df1.
# return df1.assign

In [ ]:
PERIOD_BINS = [1800, 1920, 1940, 1960, 1990, 2030]
PERIOD_LABELS = ["before1920", "1920-39", "1940-59", "1960-89", "1990+"]
feature_names = ["high-rise", "garage", "mixed-use", "service", "residential"]


def feature_types(d, high_rise):
    return [
        d["number_of_storeys"].ge(high_rise),
        d["is_garage"],
        d["is_service"] & d["upper_floor_function"].str.contains("mieszkaniowa"),
        d["is_service"],
        d["is_residential"] | d["upper_floor_function"].str.contains("mieszkaniowa"),
    ]


def add_building_features(
    df_short, df_long, high_rise=10, bins=PERIOD_BINS, labels=PERIOD_LABELS
):

    exc_func_for_service = [
        "garaże",
        "mieszkaniowa",
        "pustostan",
        "brak parteru",
        "przemysł i infrastruktura",
    ]
    data_long_features = (
        df_long.assign(
            is_garage=lambda d: d["category"] == "garaże",
            is_residential=lambda d: d["category"] == "mieszkaniowa",
            is_vacant=lambda d: d["category"] == "pustostan",
            is_service=lambda d: ~d["category"].isin(exc_func_for_service),
        )
        .groupby("id")
        .any()
        .loc[:, ["is_garage", "is_residential", "is_vacant", "is_service"]]
    )

    return df_short.join(data_long_features, validate="one_to_one").assign(
        type=lambda d: np.select(feature_types(d, high_rise), feature_names, "other"),
        time_period=lambda d: pd.cut(
            d["build_year_est"], bins, right=False, labels=labels
        ),
    )

In [ ]:
build_features = add_building_features(data_gdynia, data_gdynia_long_categories)
pd.crosstab(
    index=build_features["time_period"],
    columns=build_features["type"],
    normalize="columns",
).style.background_gradient(cmap="Blues", axis="index").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

In [ ]:
build_features

4.

In [ ]:
def street_metrics(df_long, df_short=None, min_functional=10):

    streets_long = pd.concat(
        [
            df_long.loc[lambda d: d["category"] != "brak parteru"]
            .assign(is_vacant=lambda d: d["category"] == "pustostan")
            .groupby("addr_street")
            .agg(num_premises=("category", "size"), num_vacant=("is_vacant", "sum"))
            .assign(
                num_premises_wo_vacant=lambda d: d["num_premises"] - d["num_vacant"],
                share_vacant=lambda d: d["num_vacant"].div(d["num_premises"]),
            ),
            df_long.loc[lambda d: ~d["category"].isin(["brak parteru", "pustostan"])]
            .groupby("addr_street")["category"]
            .value_counts()
            .to_frame("premise_per_cat")
            .assign(
                total_premises_per_street=lambda d: d.groupby("addr_street")[
                    "premise_per_cat"
                ].transform("sum"),
                share=lambda d: d["premise_per_cat"].div(
                    d["total_premises_per_street"]
                ),
                lnshare_mult_share=lambda d: np.log(d["share"]).mul(d["share"]).mul(-1),
            )
            .groupby("addr_street")
            .agg(
                num_cat=("premise_per_cat", lambda s: s.gt(0).sum()),
                leader_share=("share", "max"),
                func_entropy=("lnshare_mult_share", "sum"),
            )
            .assign(eff_cat=lambda d: np.exp(d["func_entropy"])),
        ],
        axis=1,
    ).loc[
        :,
        [
            "num_premises",
            "num_premises_wo_vacant",
            "num_vacant",
            "share_vacant",
            "leader_share",
            "func_entropy",
            "num_cat",
            "eff_cat",
        ],
    ]
    if df_short is not None:
        streets_long = streets_long.join(
            df_short.groupby("addr_street")
            .agg(
                num_buildings=("osm_id", "size"),
                num_mix_use=("type", lambda d: (d == "mixed-use").sum()),
            )
            .assign(share_mix_use=lambda d: d["num_mix_use"].div(d["num_buildings"]))
        )
    return streets_long.loc[lambda d: d["num_premises_wo_vacant"].ge(min_functional)]


In [ ]:
old_ids = data_gdynia.loc[
    lambda e: e["date_of_checking"] < pd.to_datetime("2026-10-22")
].index
data_gdynia_long_categories_before = data_gdynia_long_categories.loc[
    lambda d: d.index.isin(old_ids)
]


assert np.isclose(
    street_metrics(data_gdynia_long_categories_before, min_functional=0).loc[
        "Arkadiusza Rybickiego", "func_entropy"
    ],
    1.581,
    atol=0.001,
)
assert len(street_metrics(data_gdynia_long_categories_before, min_functional=10)) == 26

In [ ]:
data_gdynia_features = add_building_features(data_gdynia, data_gdynia_long_categories)
metrics_10 = street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
)

assert len(metrics_10) == 28
metrics_10.sort_values("func_entropy", ascending=False).head(2)

In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[
    :,
    [
        "num_premises_wo_vacant",
        "func_entropy",
        "share_vacant",
        "share_mix_use",
        "leader_share",
    ],
].corr(method="spearman")


In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[:, ["num_premises_wo_vacant", "func_entropy"]].corr(method="spearman")

In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[
    lambda d: d.index != "Świętojańska", ["num_premises_wo_vacant", "func_entropy"]
].corr(method="pearson")

In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[:, ["num_premises_wo_vacant", "func_entropy"]].corr(method="pearson")

5.

In [ ]:
data_gdynia_features_old = add_building_features(
    data_gdynia.loc[lambda d: d.index.isin(old_ids)],
    data_gdynia_long_categories.loc[lambda d: d.index.isin(old_ids)],
)
entropy_before_after = pd.concat(
    {
        "before": street_metrics(
            data_gdynia_long_categories_before,
            data_gdynia_features_old,
            min_functional=10,
        )
        .loc[:, ["num_premises_wo_vacant", "func_entropy"]]
        .assign(
            rank_before=lambda d: d["func_entropy"].rank(
                method="dense", ascending=False
            )
        ),
        "after": street_metrics(
            data_gdynia_long_categories, data_gdynia_features, min_functional=10
        )
        .loc[:, ["num_premises_wo_vacant", "func_entropy"]]
        .assign(
            rank_after=lambda d: d["func_entropy"].rank(method="dense", ascending=False)
        ),
    },
    axis=1,
)

In [ ]:
entropy_before_after

In [ ]:
entropy_difference_before_after = (
    entropy_before_after.dropna()
    .sort_values(("after", "rank_after"))
    .assign(
        diff_rank=lambda d: d[("before", "rank_before")] != d[("after", "rank_after")]
    )
)

In [ ]:
entropy_difference_before_after.loc[
    :, ("before", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

In [ ]:
entropy_difference_before_after.loc[
    :, ("after", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

In [ ]:
entropy_difference_before_after.loc[
    :, [("after", "func_entropy"), ("before", "func_entropy")]
].corr(method="spearman")

6.

A.

In [ ]:
tab1 = add_building_features(data_gdynia, data_gdynia_long_categories)

In [ ]:
pd.crosstab(
    index=tab1["time_period"], columns=tab1["is_vacant"], normalize="index"
).style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

B.

In [ ]:
tab2 = tab1.loc[lambda d: d["is_vacant"] | d["is_service"]]
pd.crosstab(
    index=tab2["time_period"], columns=tab2["is_vacant"], normalize="index"
).style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

C.

In [ ]:
tab3 = data_gdynia_long_categories.loc[lambda d: d["category"] != "brak parteru"].join(
    add_building_features(data_gdynia, data_gdynia_long_categories).loc[
        :, "time_period"
    ],
    validate="many_to_one",
)
pd.crosstab(
    index=tab3.reset_index().loc[:, "time_period"],
    columns=tab3.reset_index()["category"].eq("pustostan"),
    normalize="index",
).style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)